# Walmart Store Sales — TFT inference

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLfinal/Walmart-Recruiting---Store-Sales-Forecasting/blob/deep-learning/models/deep_learning/tft/tft_inference.ipynb)

This notebook performs inference only. It uses the best valid TFT experiment we kept:

```text
v7 = stable serious residual TFT + seasonal naive fallback + residual blending
best validation alpha = 0.35
```

Important: the v7 TFT was trained only on the top `2000` Store-Dept series. For test rows outside that covered set, this notebook uses the same 52-week seasonal naive fallback instead of forcing unseen categories through the TFT model.


In [ ]:
%pip install -q "torch>=2.3,<3" "lightning>=2.3,<3" "pytorch-forecasting>=1.2,<2" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "kaggle>=1.7,<2"


In [ ]:
from __future__ import annotations

import json
import os
import platform
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb

from lightning.pytorch import Trainer, seed_everything
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE

pd.set_option("display.max_columns", 120)
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "wandb": wandb.__version__,
})


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Not running in Colab or Drive unavailable: {exc}")


## Configuration

Store `WANDB_API_KEY` in Colab Secrets or log in when W&B asks. Kaggle submission is disabled by default.


In [ ]:
CONFIG = {
    "seed": 42,
    "data_dir": "/content/drive/MyDrive/walmart_competition_data",
    "output_dir": "/content/drive/MyDrive/walmart_competition_inference/tft",
    "download_dir": "/content/artifacts/tft_v7_model",
    "wandb_entity": "kende23-n-a",
    "wandb_project": "Walmart-Recruiting---Store-Sales-Forecasting",
    "model_artifact_uri": "kende23-n-a/Walmart-Recruiting---Store-Sales-Forecasting/tft-v7-stable-serious-residual-blending:experiment-v7",
    "checkpoint_filename": "tft_v7_best.ckpt",
    "submission_artifact_name": "tft-v7-kaggle-submission",
    "registry_target": "wandb-registry-model/Walmart_TFT_Model",
    "validation_weeks": 39,
    "encoder_weeks": 52,
    "top_n_series": 2000,
    "holiday_weight": 5.0,
    "batch_size": 512,
    "num_workers": 0,
    "best_blend_alpha": 0.35,
    "residual_clip": 100000.0,
    "prediction_clip_max": 300000.0,
    "submit_to_kaggle": False,
    "kaggle_competition": "walmart-recruiting-store-sales-forecasting",
    "kaggle_message": "TFT v7 residual blending alpha 0.35 with seasonal naive fallback",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

DATA_DIR = Path(CONFIG["data_dir"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
DOWNLOAD_DIR = Path(CONFIG["download_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_api_key = os.environ.get("WANDB_API_KEY")

wandb.login(key=wandb_api_key, relogin=False) if wandb_api_key else wandb.login()
seed_everything(CONFIG["seed"], workers=True)
CONFIG


## Load raw data

Inference needs `train.csv` for the 52-week seasonal baseline and encoder history. It uses `test.csv` for the actual submission rows.


In [ ]:
required_files = ["train.csv", "test.csv", "features.csv", "stores.csv"]
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError({"data_dir": str(DATA_DIR), "missing_files": missing_files})

train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features_raw = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores_raw = pd.read_csv(DATA_DIR / "stores.csv")

required_train = {"Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"}
required_test = {"Store", "Dept", "Date", "IsHoliday"}
required_features = {"Store", "Date", "Temperature", "Fuel_Price", "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5", "CPI", "Unemployment"}
required_stores = {"Store", "Type", "Size"}
missing = {
    "train": sorted(required_train.difference(train_raw.columns)),
    "test": sorted(required_test.difference(test_raw.columns)),
    "features": sorted(required_features.difference(features_raw.columns)),
    "stores": sorted(required_stores.difference(stores_raw.columns)),
}
if any(missing.values()):
    raise ValueError(missing)

train_raw = train_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
test_raw = test_raw.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)
features_raw = features_raw.sort_values(["Store", "Date"]).reset_index(drop=True)
stores_raw = stores_raw.sort_values("Store").reset_index(drop=True)

profile = pd.DataFrame({
    "table": ["train", "test", "features", "stores"],
    "rows": [len(train_raw), len(test_raw), len(features_raw), len(stores_raw)],
    "columns": [train_raw.shape[1], test_raw.shape[1], features_raw.shape[1], stores_raw.shape[1]],
    "min_date": [train_raw.Date.min(), test_raw.Date.min(), features_raw.Date.min(), pd.NaT],
    "max_date": [train_raw.Date.max(), test_raw.Date.max(), features_raw.Date.max(), pd.NaT],
})
display(profile)
display(test_raw.head())


## Rebuild the v7 top-2000 panel and seasonal baseline

The model artifact contains weights, but inference still needs the same category universe and time-series dataset schema. The top-2000 selection is deterministic: highest total train sales by Store-Dept pair.


In [ ]:
top_series = (
    train_raw.groupby(["Store", "Dept"], as_index=False)["Weekly_Sales"]
    .sum()
    .sort_values("Weekly_Sales", ascending=False)
    .head(CONFIG["top_n_series"])[["Store", "Dept"]]
)
top_series_key = set(map(tuple, top_series[["Store", "Dept"]].to_numpy()))

train_top = train_raw.merge(top_series.assign(_keep=1), on=["Store", "Dept"], how="inner").drop(columns="_keep")
test_top = test_raw.merge(top_series.assign(_keep=1), on=["Store", "Dept"], how="inner").drop(columns="_keep")

test_key = set(map(tuple, test_raw[["Store", "Dept"]].drop_duplicates().to_numpy()))
covered_key = set(map(tuple, test_top[["Store", "Dept"]].drop_duplicates().to_numpy()))
uncovered_key_count = len(test_key - covered_key)

all_train_dates = pd.Index(sorted(train_raw["Date"].unique()), name="Date")
test_dates = pd.Index(sorted(test_raw["Date"].unique()), name="Date")
combined_dates = pd.Index(sorted(set(all_train_dates).union(set(test_dates))), name="Date")

if len(test_dates) != CONFIG["validation_weeks"]:
    raise ValueError(f"Expected test horizon {CONFIG['validation_weeks']}, got {len(test_dates)}")
if len(all_train_dates) < CONFIG["encoder_weeks"]:
    raise ValueError("Not enough train history for encoder window.")

combined_date_to_idx = {date: idx for idx, date in enumerate(combined_dates)}
test_start_idx = int(combined_date_to_idx[test_dates.min()])

print({
    "top_n_series": CONFIG["top_n_series"],
    "train_rows_before": len(train_raw),
    "train_rows_after_top_filter": len(train_top),
    "test_rows_total": len(test_raw),
    "test_rows_tft_covered": len(test_top),
    "test_series_total": len(test_key),
    "test_series_tft_covered": len(covered_key),
    "test_series_fallback_only": uncovered_key_count,
    "train_date_range": (str(all_train_dates.min().date()), str(all_train_dates.max().date())),
    "test_date_range": (str(test_dates.min().date()), str(test_dates.max().date())),
    "test_start_idx": test_start_idx,
})


In [ ]:
COVARIATE_REALS = [
    "Temperature", "Fuel_Price", "MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5",
    "CPI", "Unemployment", "Size"
]
CALENDAR_REALS = ["time_idx", "week_sin", "week_cos", "month_sin", "month_cos"]
KNOWN_REALS = CALENDAR_REALS + COVARIATE_REALS
TARGET_COL = "ResidualSales"
SEASONAL_COL = "SeasonalNaive52"


def build_combined_sales_panel(train_df: pd.DataFrame, index_pairs: pd.MultiIndex | None = None) -> pd.DataFrame:
    panel = (
        train_df.pivot_table(index=["Store", "Dept"], columns="Date", values="Weekly_Sales", aggfunc="sum")
        .reindex(columns=combined_dates)
        .fillna(0.0)
        .sort_index()
    )
    if index_pairs is not None:
        panel = panel.reindex(index=index_pairs).fillna(0.0)
    return panel


full_sales_panel = build_combined_sales_panel(train_raw)
top_sales_panel = build_combined_sales_panel(train_top)
full_seasonal_panel = full_sales_panel.shift(52, axis=1).fillna(0.0)
top_seasonal_panel = top_sales_panel.shift(52, axis=1).fillna(0.0)


def seasonal_lookup_from_panel(panel: pd.DataFrame) -> pd.DataFrame:
    lookup = panel.stack().rename(SEASONAL_COL).reset_index()
    lookup["Store"] = lookup["Store"].astype(int)
    lookup["Dept"] = lookup["Dept"].astype(int)
    lookup["Date"] = pd.to_datetime(lookup["Date"])
    return lookup


top_seasonal_lookup_long = seasonal_lookup_from_panel(top_seasonal_panel)
full_seasonal_lookup_long = seasonal_lookup_from_panel(full_seasonal_panel)


def add_features(out: pd.DataFrame) -> pd.DataFrame:
    feat = features_raw.drop(columns=[c for c in ["IsHoliday"] if c in features_raw.columns])
    out = out.merge(feat, on=["Store", "Date"], how="left")
    out = out.merge(stores_raw, on="Store", how="left")

    markdown_cols = ["MarkDown1", "MarkDown2", "MarkDown3", "MarkDown4", "MarkDown5"]
    out[markdown_cols] = out[markdown_cols].fillna(0.0)
    for col in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
        out[col] = out.groupby("Store")[col].transform(lambda s: s.ffill().bfill())
        out[col] = out[col].fillna(out[col].median())
    out["Size"] = out["Size"].fillna(out["Size"].median())
    out["Type"] = out["Type"].fillna("Unknown")

    out["time_idx"] = out["Date"].map(combined_date_to_idx).astype("int64")
    iso_week = out["Date"].dt.isocalendar().week.astype(int)
    month = out["Date"].dt.month.astype(int)
    out["week_sin"] = np.sin(2 * np.pi * iso_week / 52.0)
    out["week_cos"] = np.cos(2 * np.pi * iso_week / 52.0)
    out["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    out["month_cos"] = np.cos(2 * np.pi * month / 12.0)

    out["Store"] = out["Store"].astype(str)
    out["Dept"] = out["Dept"].astype(str)
    out["Type"] = out["Type"].astype(str)
    out["IsHoliday"] = out["IsHoliday"].astype(int).astype(str)
    for col in KNOWN_REALS:
        if col != "time_idx":
            out[col] = out[col].astype(float)
    out["time_idx"] = out["time_idx"].astype("int64")
    return out


def make_train_frame(df: pd.DataFrame, seasonal_lookup: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["Store", "Dept", "Date"]).copy()
    out["Weekly_Sales_Original"] = out["Weekly_Sales"].astype(float)
    out["Weekly_Sales_Clipped"] = out["Weekly_Sales_Original"].clip(lower=0.0)
    out = out.merge(seasonal_lookup, on=["Store", "Dept", "Date"], how="left")
    out[SEASONAL_COL] = out[SEASONAL_COL].fillna(0.0).astype(float)
    out[TARGET_COL] = (out["Weekly_Sales_Clipped"] - out[SEASONAL_COL]).astype(float)
    out[TARGET_COL] = out[TARGET_COL].clip(-CONFIG["residual_clip"], CONFIG["residual_clip"])
    return add_features(out)


def make_future_frame(df: pd.DataFrame, seasonal_lookup: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["Store", "Dept", "Date"]).copy()
    out = out.merge(seasonal_lookup, on=["Store", "Dept", "Date"], how="left")
    out[SEASONAL_COL] = out[SEASONAL_COL].fillna(0.0).astype(float)
    out["Weekly_Sales_Original"] = np.nan
    out["Weekly_Sales_Clipped"] = np.nan
    # Future target is unknown. The value is only a placeholder required by TimeSeriesDataSet.
    out[TARGET_COL] = 0.0
    return add_features(out)


train_frame = make_train_frame(train_top, top_seasonal_lookup_long)
future_frame = make_future_frame(test_top, top_seasonal_lookup_long)
predict_frame = pd.concat([train_frame, future_frame], ignore_index=True).sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

print({
    "train_frame_rows": len(train_frame),
    "future_frame_rows": len(future_frame),
    "predict_frame_rows": len(predict_frame),
    "n_tft_series": int(train_frame[["Store", "Dept"]].drop_duplicates().shape[0]),
    "known_reals": KNOWN_REALS,
    "future_seasonal_min": float(future_frame[SEASONAL_COL].min()) if len(future_frame) else 0.0,
    "future_seasonal_mean": float(future_frame[SEASONAL_COL].mean()) if len(future_frame) else 0.0,
    "future_seasonal_max": float(future_frame[SEASONAL_COL].max()) if len(future_frame) else 0.0,
})
display(future_frame[["Store", "Dept", "Date", SEASONAL_COL, "IsHoliday"] + KNOWN_REALS[:5]].head())


## Rebuild dataset schema and download v7 checkpoint

The dataset is rebuilt from the same preprocessing rules. The checkpoint itself is downloaded from W&B.


In [ ]:
fit_dates = all_train_dates[:-CONFIG["validation_weeks"]]
split_pos = len(fit_dates)
training_cutoff = split_pos - 1
# Match v7 training exactly: fit encoders/scalers on the pre-validation period only.
# Prediction still uses all train rows as encoder history plus test rows as decoder horizon.

training_dataset = TimeSeriesDataSet(
    train_frame[train_frame.time_idx <= training_cutoff],
    time_idx="time_idx",
    min_prediction_idx=52,
    target=TARGET_COL,
    group_ids=["Store", "Dept"],
    min_encoder_length=CONFIG["encoder_weeks"] // 2,
    max_encoder_length=CONFIG["encoder_weeks"],
    min_prediction_length=CONFIG["validation_weeks"],
    max_prediction_length=CONFIG["validation_weeks"],
    static_categoricals=["Store", "Dept", "Type"],
    time_varying_known_categoricals=["IsHoliday"],
    time_varying_known_reals=KNOWN_REALS,
    time_varying_unknown_reals=[TARGET_COL],
    target_normalizer=GroupNormalizer(groups=["Store", "Dept"], center=True),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

prediction_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    predict_frame,
    predict=True,
    stop_randomization=True,
    min_prediction_idx=test_start_idx,
)
pred_loader = prediction_dataset.to_dataloader(
    train=False,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
)

print({
    "split_pos": split_pos,
    "training_cutoff": training_cutoff,
    "fit_range": (str(fit_dates.min().date()), str(fit_dates.max().date())),
    "test_start_idx": test_start_idx,
    "prediction_samples": len(prediction_dataset),
    "prediction_batches": len(pred_loader),
})



In [ ]:
inference_run = wandb.init(
    entity=CONFIG["wandb_entity"],
    project=CONFIG["wandb_project"],
    name="tft_v7_inference_residual_blending",
    group="tft-inference",
    job_type="inference",
    tags=["tft", "inference", "residual-blending", "seasonal-naive-fallback", "v7"],
    config=CONFIG,
    save_code=True,
)

model_artifact = inference_run.use_artifact(CONFIG["model_artifact_uri"], type="model")
artifact_dir = Path(model_artifact.download(root=str(DOWNLOAD_DIR)))
checkpoint_path = artifact_dir / CONFIG["checkpoint_filename"]
if not checkpoint_path.exists():
    candidates = sorted(artifact_dir.glob("*.ckpt"))
    if not candidates:
        raise FileNotFoundError(f"No checkpoint found in {artifact_dir}")
    checkpoint_path = candidates[0]

print({
    "artifact": CONFIG["model_artifact_uri"],
    "artifact_dir": str(artifact_dir),
    "checkpoint_path": str(checkpoint_path),
})


## Predict residuals and reconstruct sales

For covered rows:

```text
prediction = SeasonalNaive52 + 0.35 * PredictedResidual
```

For uncovered Store-Dept rows, prediction is the 52-week seasonal naive fallback.


In [ ]:
best_tft = TemporalFusionTransformer.load_from_checkpoint(str(checkpoint_path), loss=MAE())
best_tft.eval()

prediction = best_tft.predict(
    pred_loader,
    mode="prediction",
    return_index=True,
    trainer_kwargs={"accelerator": "gpu" if torch.cuda.is_available() else "cpu", "devices": 1},
)

pred_residual = prediction.output.detach().cpu().numpy() if torch.is_tensor(prediction.output) else np.asarray(prediction.output)
index_df = prediction.index.reset_index(drop=True).copy()
if pred_residual.ndim == 3:
    pred_residual = pred_residual[..., 0]
if pred_residual.shape[1] != CONFIG["validation_weeks"]:
    raise ValueError(f"Expected horizon {CONFIG['validation_weeks']}, got {pred_residual.shape}")

finite_mask = np.isfinite(pred_residual)
finite_values = pred_residual[finite_mask]
prediction_diagnostics = {
    "raw_prediction_nan_count": int(np.isnan(pred_residual).sum()),
    "raw_prediction_posinf_count": int(np.isposinf(pred_residual).sum()),
    "raw_prediction_neginf_count": int(np.isneginf(pred_residual).sum()),
    "raw_prediction_finite_count": int(finite_mask.sum()),
    "raw_prediction_total_count": int(pred_residual.size),
    "raw_prediction_min_finite": float(np.min(finite_values)) if finite_values.size else np.nan,
    "raw_prediction_mean_finite": float(np.mean(finite_values)) if finite_values.size else np.nan,
    "raw_prediction_max_finite": float(np.max(finite_values)) if finite_values.size else np.nan,
}
print(prediction_diagnostics)

pred_residual = np.nan_to_num(pred_residual, nan=0.0, posinf=0.0, neginf=0.0)
pred_residual = np.clip(pred_residual, -CONFIG["residual_clip"], CONFIG["residual_clip"])
prediction_diagnostics.update({
    "postprocessed_prediction_min": float(np.min(pred_residual)),
    "postprocessed_prediction_mean": float(np.mean(pred_residual)),
    "postprocessed_prediction_max": float(np.max(pred_residual)),
})

records = []
for row_idx, row in index_df.iterrows():
    store = int(row["Store"])
    dept = int(row["Dept"])
    for horizon_idx, date in enumerate(test_dates):
        seasonal_base = float(top_seasonal_panel.loc[(store, dept), date])
        residual_pred = float(pred_residual[row_idx, horizon_idx])
        blended_pred = float(np.clip(
            seasonal_base + CONFIG["best_blend_alpha"] * residual_pred,
            0.0,
            CONFIG["prediction_clip_max"],
        ))
        records.append({
            "Store": store,
            "Dept": dept,
            "Date": pd.Timestamp(date),
            "SeasonalNaive52": seasonal_base,
            "PredictedResidual": residual_pred,
            "BlendAlpha": CONFIG["best_blend_alpha"],
            "TFTPrediction": blended_pred,
        })

tft_pred_df = pd.DataFrame(records)
if tft_pred_df.empty:
    raise ValueError("No TFT predictions were created.")
if not np.isfinite(tft_pred_df[["SeasonalNaive52", "PredictedResidual", "TFTPrediction"]].to_numpy()).all():
    raise ValueError("Non-finite values remain in TFT predictions.")

print({
    "tft_prediction_rows": len(tft_pred_df),
    "tft_prediction_min": float(tft_pred_df["TFTPrediction"].min()),
    "tft_prediction_mean": float(tft_pred_df["TFTPrediction"].mean()),
    "tft_prediction_max": float(tft_pred_df["TFTPrediction"].max()),
    **prediction_diagnostics,
})
display(tft_pred_df.head())


In [ ]:
# Full-test fallback from the same 52-week seasonal lookup.
fallback_df = (
    test_raw[["Store", "Dept", "Date", "IsHoliday"]]
    .merge(full_seasonal_lookup_long, on=["Store", "Dept", "Date"], how="left")
)
fallback_df[SEASONAL_COL] = fallback_df[SEASONAL_COL].fillna(0.0).astype(float)
fallback_df["FallbackPrediction"] = fallback_df[SEASONAL_COL].clip(0.0, CONFIG["prediction_clip_max"])

submission_frame = fallback_df.merge(
    tft_pred_df[["Store", "Dept", "Date", "TFTPrediction", "PredictedResidual", "BlendAlpha"]],
    on=["Store", "Dept", "Date"],
    how="left",
)
submission_frame["UsedTFT"] = submission_frame["TFTPrediction"].notna()
submission_frame["Weekly_Sales"] = np.where(
    submission_frame["UsedTFT"],
    submission_frame["TFTPrediction"],
    submission_frame["FallbackPrediction"],
)
submission_frame["Weekly_Sales"] = submission_frame["Weekly_Sales"].clip(0.0, CONFIG["prediction_clip_max"])
submission_frame.insert(
    0,
    "Id",
    submission_frame["Store"].astype(str) + "_" + submission_frame["Dept"].astype(str) + "_" + submission_frame["Date"].dt.strftime("%Y-%m-%d"),
)
submission = submission_frame[["Id", "Weekly_Sales"]].copy()

expected_ids = test_raw["Store"].astype(str) + "_" + test_raw["Dept"].astype(str) + "_" + test_raw["Date"].dt.strftime("%Y-%m-%d")
if not submission["Id"].equals(expected_ids.reset_index(drop=True)):
    raise ValueError("Submission row order does not match test.csv order.")
if submission["Weekly_Sales"].isna().any():
    raise ValueError("Submission contains NaN predictions.")

coverage = {
    "submission_rows": int(len(submission)),
    "tft_rows": int(submission_frame["UsedTFT"].sum()),
    "fallback_rows": int((~submission_frame["UsedTFT"]).sum()),
    "tft_row_coverage": float(submission_frame["UsedTFT"].mean()),
    "prediction_min": float(submission["Weekly_Sales"].min()),
    "prediction_mean": float(submission["Weekly_Sales"].mean()),
    "prediction_max": float(submission["Weekly_Sales"].max()),
}
print(coverage)
display(submission.head())
display(submission_frame.head())


## Save and log inference artifacts


In [ ]:
submission_path = OUTPUT_DIR / "tft_v7_submission.csv"
detailed_predictions_path = OUTPUT_DIR / "tft_v7_detailed_predictions.csv"
tft_predictions_path = OUTPUT_DIR / "tft_v7_tft_covered_predictions.csv"
manifest_path = OUTPUT_DIR / "tft_v7_inference_manifest.json"

submission.to_csv(submission_path, index=False)
submission_frame.to_csv(detailed_predictions_path, index=False)
tft_pred_df.to_csv(tft_predictions_path, index=False)

manifest = {
    "model": "TemporalFusionTransformer",
    "source_experiment": "tft_v7_stable_serious_residual_blending",
    "model_artifact_uri": CONFIG["model_artifact_uri"],
    "checkpoint_path": str(checkpoint_path),
    "target_strategy": "seasonal_residual_52w_clipped",
    "postprocess_strategy": "residual_blending_with_seasonal_naive_fallback",
    "blend_alpha": CONFIG["best_blend_alpha"],
    "top_n_series": CONFIG["top_n_series"],
    "encoder_weeks": CONFIG["encoder_weeks"],
    "prediction_diagnostics": prediction_diagnostics,
    **coverage,
}
manifest_path.write_text(json.dumps(manifest, indent=2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(submission["Weekly_Sales"], bins=100)
ax.set_title("TFT v7 inference prediction distribution")
ax.set_xlabel("Weekly_Sales")
ax.set_ylabel("count")
plt.tight_layout()
hist_path = OUTPUT_DIR / "tft_v7_prediction_histogram.png"
fig.savefig(hist_path, dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
coverage_df = pd.DataFrame({
    "source": ["TFT covered", "Seasonal fallback"],
    "rows": [coverage["tft_rows"], coverage["fallback_rows"]],
})
ax.bar(coverage_df["source"], coverage_df["rows"])
ax.set_title("TFT v7 inference row coverage")
ax.set_ylabel("rows")
plt.tight_layout()
coverage_plot_path = OUTPUT_DIR / "tft_v7_coverage.png"
fig.savefig(coverage_plot_path, dpi=160)
plt.show()

artifact = wandb.Artifact(
    CONFIG["submission_artifact_name"],
    type="inference-output",
    description="TFT v7 residual blending Kaggle submission with seasonal naive fallback.",
    metadata=manifest,
)
artifact.add_file(str(submission_path))
artifact.add_file(str(detailed_predictions_path))
artifact.add_file(str(tft_predictions_path))
artifact.add_file(str(manifest_path))
artifact.add_file(str(hist_path))
artifact.add_file(str(coverage_plot_path))
inference_run.log_artifact(artifact, aliases=["tft-v7", "latest", "submission"])

# Link the already-used model artifact to a registry collection if permissions allow it.
try:
    inference_run.link_artifact(model_artifact, CONFIG["registry_target"], aliases=["tft-v7", "latest"])
    registry_status = "linked"
except Exception as exc:
    registry_status = f"skipped: {exc}"

for key, value in manifest.items():
    if isinstance(value, (int, float, str, bool)) or value is None:
        inference_run.summary[key] = value
for key, value in prediction_diagnostics.items():
    inference_run.summary[key] = value
inference_run.summary["registry_status"] = registry_status

wandb.log({
    "inference/submission_rows": coverage["submission_rows"],
    "inference/tft_rows": coverage["tft_rows"],
    "inference/fallback_rows": coverage["fallback_rows"],
    "inference/tft_row_coverage": coverage["tft_row_coverage"],
    "inference/prediction_min": coverage["prediction_min"],
    "inference/prediction_mean": coverage["prediction_mean"],
    "inference/prediction_max": coverage["prediction_max"],
    "inference/submission_sample": wandb.Table(dataframe=submission.head(1000)),
    "inference/detailed_prediction_sample": wandb.Table(dataframe=submission_frame.sample(min(20000, len(submission_frame)), random_state=CONFIG["seed"])),
    "inference/prediction_histogram": wandb.Image(str(hist_path)),
    "inference/coverage_plot": wandb.Image(str(coverage_plot_path)),
    **{f"prediction_diagnostics/{k}": v for k, v in prediction_diagnostics.items()},
})

print({
    "submission_path": str(submission_path),
    "detailed_predictions_path": str(detailed_predictions_path),
    "manifest_path": str(manifest_path),
    "registry_status": registry_status,
})
manifest


## Optional Kaggle submission

This cell is disabled by default. Set `CONFIG["submit_to_kaggle"] = True` only when you intentionally want to submit.


In [ ]:
if CONFIG["submit_to_kaggle"]:
    cmd = [
        "kaggle", "competitions", "submit",
        "-c", CONFIG["kaggle_competition"],
        "-f", str(submission_path),
        "-m", CONFIG["kaggle_message"],
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError("Kaggle submission failed")
else:
    print("Kaggle submission skipped. Set CONFIG['submit_to_kaggle'] = True to submit.")


In [ ]:
wandb.finish()
